# 第二阶段 · Notebook 1：VQE原理与实现

> **配合阅读**：  
> - `acs.chemrev.8b00803.pdf` 第IV章：Variational methods  
> - `RevModPhys.92.015003.pdf` 第V章：Variational quantum algorithms  
> - `science.abd3880.pdf`：Google VQE实验视角

---

## 本节目标
1. 深入理解VQE算法的数学基础与收敛保证
2. 掌握UCCSD、硬件高效(HEA)、ADAPT-VQE等主流ansatz设计策略
3. 理解量子相位估计（QPE）并与VQE对比
4. 掌握误差缓解的基本技术（ZNE）

---
## 1. VQE的数学基础：变分定理

### 1.1 Rayleigh-Ritz变分原理

$$E_0 \leq \frac{\langle \psi(\theta) | \hat{H} | \psi(\theta) \rangle}{\langle \psi(\theta) | \psi(\theta) \rangle}$$

**意义**：对任意归一化试探态 $|\psi(\theta)\rangle$，其能量期望值**永远不低于真实基态能量**。  
因此，最小化期望值就等价于寻找最佳近似基态。

你在DFT中已经用过这个原理——Kohn-Sham方程就是在单行列式近似下最小化总能量泛函！

### 1.2 VQE vs 经典变分方法

| 方法 | 试探态形式 | 变分参数 | 计算量 |
|------|----------|---------|--------|
| HF | 单Slater行列式 | 轨道系数 | O(N³) |
| CCSD(T) | CC展开 | t₁, t₂振幅 | O(N⁷) |
| VQE | 参数化量子电路 | θ₁,…,θₘ | 经典O(M)，量子O(深度) |
| FCI | 全CI展开 | 所有CI系数 | O(exp(N)) |

VQE的优势：利用量子叠加和纠缠，在量子硬件上高效表达强关联波函数。

---
## 2. UCCSD Ansatz 深入分析

### 2.1 经典CCSD回顾

$$|\psi_{CCSD}\rangle = e^{T_1 + T_2} |\phi_0\rangle$$

其中：
- $T_1 = \sum_{ia} t_i^a \hat{a}_a^\dagger \hat{a}_i$（单激发）
- $T_2 = \sum_{ijab} t_{ij}^{ab} \hat{a}_a^\dagger \hat{a}_b^\dagger \hat{a}_j \hat{a}_i$（双激发）

### 2.2 UCCSD（酉耦合簇）

$$|\psi_{UCCSD}(\theta)\rangle = e^{\hat{T}(\theta) - \hat{T}^\dagger(\theta)} |\phi_0\rangle$$

**关键区别**：经典CC是非酉的（不对应量子电路）；UCC是**酉变换**，可以直接在量子计算机上实现！

JW变换后，每个激发算符 $\hat{a}_p^\dagger \hat{a}_q$ 变为泡利串，指数映射变为一系列旋转门（Pauli exponentials）。

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# ─── 2.3 手动实现UCCSD单激发门 ───
# 单激发激发算符 T₁ = t·(a†_2 a_0 - a†_0 a_2)
# JW变换后：T₁ ∝ θ·(X₀Y₁ - Y₀X₁)/2  （对于2轨道系统）
# 对应量子电路：e^{iθ(XY-YX)/2}

def givens_rotation(n_qubits, p, q, theta):
    """
    构造 Givens 旋转电路，实现单激发的酉演化
    等价于 e^{iθ(X_p Y_q - Y_p X_q)/2}
    """
    qc = QuantumCircuit(n_qubits)
    
    # 线路分解（Givens旋转的标准分解）
    qc.cx(q, p)
    qc.ry(theta / 2, q)
    qc.cx(p, q)
    qc.ry(-theta / 2, q)
    qc.cx(p, q)
    qc.cx(q, p)
    
    return qc

# 双激发：e^{iθ(XXYX + YXXX - XYXX - XXXY - YYXY - XYYY + YXYY + YYYX)/8}
# 电路实现更复杂，通常由Qiskit Nature自动生成

# ─── 完整的H₂ UCCSD Ansatz（手动构造）───
t1 = Parameter('t1')   # 单激发振幅
t2 = Parameter('t2')   # 双激发振幅

qc_uccsd = QuantumCircuit(4)  # H₂: 4自旋轨道

# 初始化为HF参考态 |1100⟩（自旋轨道0和1被占据）
qc_uccsd.x([0, 1])

# 单激发：α电子从轨道0→2（空间轨道0→1，α自旋）
qc_uccsd.compose(givens_rotation(4, 0, 2, t1), inplace=True)

# 双激发：αβ → αβ 双激发（轨道0,1→轨道2,3）
# 简化版本（完整版需要更多CNOT门）
qc_uccsd.cx(0, 2)
qc_uccsd.cx(1, 3)
qc_uccsd.ry(t2, 0)
qc_uccsd.cx(0, 2)
qc_uccsd.cx(1, 3)

print("H₂ UCCSD Ansatz 电路：")
print(qc_uccsd.draw('text'))
print(f"\n参数数量: {qc_uccsd.num_parameters}")
print(f"电路深度: {qc_uccsd.depth()}")
print(f"双量子比特门数: {qc_uccsd.num_nonlocal_gates()}")

---
## 3. VQE优化器策略

### 常用优化器对比

| 优化器 | 类型 | 特点 | 适用场景 |
|--------|------|------|----------|
| COBYLA | 梯度-free | 稳健、无需梯度，但收敛慢 | 少参数(<20)，首选 |
| SPSA | 随机近似 | 耗噪声友好，每次仅需2次电路评估 | 噪声设备 |
| L-BFGS-B | 基于梯度 | 快速收敛，需要梯度 | 模拟器(精确梯度) |
| Adam | 自适应 | 深度学习常用，适合多参数 | 大参数量，QML |
| Parameter Shift | 量子梯度 | 精确量子梯度（∂E/∂θ = [E(θ+π/2)-E(θ-π/2)]/2） | 真实量子设备 |

### Parameter Shift Rule（量子梯度公式）

$$\frac{\partial E(\theta)}{\partial \theta_k} = \frac{E(\theta_k + \pi/2) - E(\theta_k - \pi/2)}{2}$$

这是量子计算特有的精确梯度公式，**不需要有限差分近似**！

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt

# H₂ 泡利哈密顿量
H2 = SparsePauliOp.from_list([
    ('II', -1.8505), ('IZ', 0.3980), ('ZI', -0.3980),
    ('ZZ', 0.0112), ('XX', 0.1807), ('YY', 0.1807),
])

# ─── 3.1 Parameter Shift 梯度计算 ───
estimator = StatevectorEstimator()

def compute_gradient_parameter_shift(circuit, hamiltonian, params, param_idx):
    """Parameter Shift Rule计算单参数梯度"""
    shift = np.pi / 2
    params_plus = params.copy()
    params_plus[param_idx] += shift
    params_minus = params.copy()
    params_minus[param_idx] -= shift
    
    pub_plus = (circuit, hamiltonian, [params_plus])
    pub_minus = (circuit, hamiltonian, [params_minus])
    results = estimator.run([pub_plus, pub_minus]).result()
    
    E_plus = float(results[0].data.evs[0])
    E_minus = float(results[1].data.evs[0])
    return (E_plus - E_minus) / 2

# ─── 3.2 对比不同优化器 ───
theta = Parameter('θ')
ansatz = QuantumCircuit(2)
ansatz.x(0)
ansatz.ry(2 * theta, 0)
ansatz.cx(0, 1)

results_by_optimizer = {}

for method in ['COBYLA', 'L-BFGS-B', 'SLSQP']:
    history = []
    
    def cost_with_history(params):
        pub = (ansatz, H2, [params])
        E = float(estimator.run([pub]).result()[0].data.evs[0])
        history.append(E)
        return E
    
    np.random.seed(0)
    x0 = np.random.uniform(0, np.pi, 1)
    
    if method in ['L-BFGS-B', 'SLSQP']:
        # 有限差分梯度
        res = minimize(cost_with_history, x0, method=method, 
                      options={'maxiter': 50})
    else:
        res = minimize(cost_with_history, x0, method=method,
                      options={'maxiter': 100, 'rhobeg': 0.5})
    
    results_by_optimizer[method] = {
        'history': history,
        'final_energy': res.fun,
        'n_evals': len(history)
    }

# 绘图对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {'COBYLA': 'blue', 'L-BFGS-B': 'green', 'SLSQP': 'orange'}
for method, data in results_by_optimizer.items():
    axes[0].plot(data['history'], color=colors[method], 
                 label=f"{method} ({data['n_evals']}次评估)", linewidth=1.5)

axes[0].axhline(y=-1.8572, color='r', linestyle='--', label='FCI参考值')
axes[0].set_xlabel('函数评估次数')
axes[0].set_ylabel('能量 (Hartree)')
axes[0].set_title('不同优化器的VQE收敛对比')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 最终误差对比（柱状图）
methods = list(results_by_optimizer.keys())
errors = [abs(results_by_optimizer[m]['final_energy'] - (-1.8572)) * 1000 for m in methods]
axes[1].bar(methods, errors, color=[colors[m] for m in methods], alpha=0.7)
axes[1].axhline(y=1.6, color='r', linestyle='--', label='化学精度 (1.6 mHa)')
axes[1].set_ylabel('误差 (mHa)')
axes[1].set_title('各优化器最终误差')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('VQE_optimizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n化学精度标准: 1 kcal/mol ≈ 1.6 mHa")
for method, data in results_by_optimizer.items():
    err_mHa = abs(data['final_energy'] - (-1.8572)) * 1000
    status = "✓ 达到化学精度" if err_mHa < 1.6 else "✗ 未达化学精度"
    print(f"  {method}: {err_mHa:.3f} mHa  {status}")

---
## 4. 量子相位估计（QPE）

### 4.1 QPE原理

QPE是**容错量子计算**（Fault-Tolerant QC）时代的核心算法，可以精确测量哈密顿量的特征值。

**算法流程**：
1. 制备辅助寄存器（m个量子比特）于均匀叠加态
2. 制备系统寄存器于近似基态 $|\psi\rangle$
3. 施加受控酉演化 $e^{-iHt}$（哈密顿量模拟）
4. 逆量子傅里叶变换
5. 测量辅助寄存器读出能量 $E_0 = \phi \cdot 2\pi / t$

$$U|\psi_0\rangle = e^{-iE_0 t}|\psi_0\rangle \implies \text{QPE测量} \rightarrow E_0$$

### 4.2 VQE vs QPE 比较

| 特性 | VQE | QPE |
|------|-----|-----|
| 量子设备要求 | NISQ（当前可用） | 容错量子计算机（未来）|
| 电路深度 | 浅（~100-1000门） | 深（~10¹⁰门，大体系）|
| 精度限制 | Ansatz表达能力 | 辅助比特数 |
| 经典计算开销 | 大（经典优化） | 小 |
| 量子优势 | 有争议 | 理论上确定 |
| 适用场景 | 当前NISQ+近期 | 未来10-15年 |

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# ─── 4.3 简化QPE演示（单量子比特哈密顿量）───
# 演示QPE原理：测量 H = E|ψ⟩⟨ψ| 的本征值 E

# 目标：测量 Z 算符的本征值（E = +1 或 -1）

def simple_qpe(n_ancilla=3, eigenstate_is_1=False):
    """
    简化QPE：测量 Z 算符的本征值
    Z|0⟩ = |0⟩ (E=1), Z|1⟩ = -|1⟩ (E=-1)
    编码为相位：U|ψ⟩ = e^{2πiφ}|ψ⟩, φ=0 (E=+1) 或 φ=0.5 (E=-1)
    """
    n_total = n_ancilla + 1
    qc = QuantumCircuit(n_total, n_ancilla)
    
    # 系统量子比特（最后一个）初始化为本征态
    if eigenstate_is_1:
        qc.x(n_total - 1)  # |1⟩ 是 Z 的本征值 -1 的本征态
    # else: |0⟩ 是 Z 的本征值 +1 的本征态
    
    # 辅助比特制备为均匀叠加
    for i in range(n_ancilla):
        qc.h(i)
    
    # 受控 Z 门（哈密顿量模拟 e^{-iZt} 的离散版本）
    for i in range(n_ancilla):
        # 受控相位门（模拟受控哈密顿量演化）
        qc.cp(2 * np.pi / 2**(n_ancilla - i), i, n_total - 1)
    
    # 逆量子傅里叶变换
    for j in range(n_ancilla // 2):
        qc.swap(j, n_ancilla - j - 1)
    for j in range(n_ancilla):
        for k in range(j):
            qc.cp(-np.pi / 2**(j - k), k, j)
        qc.h(j)
    
    qc.measure(range(n_ancilla), range(n_ancilla))
    return qc

print("QPE概念演示电路（3辅助比特，测量Z算符本征值）：")
qc_qpe = simple_qpe(n_ancilla=3, eigenstate_is_1=True)
print(qc_qpe.draw('text'))

print("\n=== QPE vs VQE 核心区别 ===")
print("VQE: 用量子电路制备近似基态，然后直接测量⟨H⟩")
print("     → 精度受ansatz限制，需要经典优化循环")
print("QPE: 编码时间演化 e^{-iHt}，通过相位读出精确能量")
print("     → 精度可达任意精度，需要深量子电路（容错QC）")
print()
print("关键公式：e^{-iHt}|E_0⟩ = e^{-iE_0 t}|E_0⟩")
print("QPE从相位 φ = E_0·t/(2π) 中提取能量 E_0")

---
## 5. 误差缓解：零噪声外推（ZNE）

在没有量子纠错的NISQ设备上，需要使用**误差缓解（Error Mitigation）**技术从有噪声的结果中提取精确值。

### Zero-Noise Extrapolation (ZNE)

核心思想：在**人为增大噪声率**的多个噪声水平下运行，然后外推到零噪声极限。

$$E_{\text{true}} \approx E_{\lambda \to 0} = \text{extrapolate}(E(\lambda_1), E(\lambda_2), E(\lambda_3), \ldots)$$

**噪声放大方式**：
- **门折叠（Gate Folding）**：将电路中的门替换为 $G \to G G^\dagger G$（相同效果，但3倍噪声）
- **Pauli Twirling**：随机插入泡利门，将相干噪声转化为去极化噪声

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

# ─── 5.1 ZNE演示 ───
# 真实VQE能量（模拟器，无噪声）
E_true = -1.8572  # FCI参考值 (Ha)

# 模拟不同噪声级别下的VQE能量（用参数化噪声模型模拟"实验结果"）
base_error_rate = 0.005  # 基础误差率 0.5%
scale_factors = [1, 2, 3, 5]  # 噪声放大因子

# H₂ ansatz（最优参数）
from qiskit.circuit import Parameter
theta = Parameter('θ')
ansatz = QuantumCircuit(2)
ansatz.x(0)
ansatz.ry(2 * theta, 0)
ansatz.cx(0, 1)

H2 = SparsePauliOp.from_list([
    ('II', -1.8505), ('IZ', 0.3980), ('ZI', -0.3980),
    ('ZZ', 0.0112), ('XX', 0.1807), ('YY', 0.1807),
])

# 先找到最优参数
from scipy.optimize import minimize
estimator_clean = StatevectorEstimator()

def cost(p):
    pub = (ansatz, H2, [p])
    return float(estimator_clean.run([pub]).result()[0].data.evs[0])

result = minimize(cost, [0.5], method='COBYLA')
theta_opt = result.x[0]
print(f"最优参数: θ = {np.degrees(theta_opt):.2f}°")
print(f"理想VQE能量: {result.fun:.6f} Ha")

# 模拟噪声效应（用解析模型代替真实量子设备）
# 噪声导致期望值衰减：E_noisy(λ) ≈ E_true + λ·ε + O(λ²)
noisy_energies = []
for scale in scale_factors:
    # 模拟去极化噪声效应：每个双量子比特门引入误差
    error = scale * base_error_rate * abs(E_true - 0)  # 简化噪声模型
    E_noisy = result.fun + error  # 噪声使能量偏高
    noisy_energies.append(E_noisy)

# ZNE外推（多项式拟合）
# 线性外推
coeffs_linear = np.polyfit(scale_factors[:2], noisy_energies[:2], 1)
E_zne_linear = np.polyval(coeffs_linear, 0)

# 二次外推
coeffs_quad = np.polyfit(scale_factors[:3], noisy_energies[:3], 2)
E_zne_quad = np.polyval(coeffs_quad, 0)

# Richardson外推（指数型）
E_zne_rich = (4 * noisy_energies[1] - noisy_energies[0]) / 3  # 1阶Richardson

# 可视化
lambda_range = np.linspace(0, 5.5, 100)
plt.figure(figsize=(9, 5))
plt.scatter(scale_factors, noisy_energies, s=80, zorder=5, color='blue', label='有噪声测量值')
plt.plot(lambda_range, np.polyval(coeffs_linear, lambda_range), 'g--', 
         label=f'线性外推: E(0)={E_zne_linear:.4f} Ha')
plt.plot(lambda_range, np.polyval(coeffs_quad, lambda_range), 'orange', linestyle='-.', 
         label=f'二次外推: E(0)={E_zne_quad:.4f} Ha')
plt.scatter([0], [E_true], s=100, color='red', zorder=6, label=f'真实值: {E_true:.4f} Ha', marker='*')
plt.xlabel('噪声放大因子 λ')
plt.ylabel('能量 (Hartree)')
plt.title('零噪声外推（ZNE）误差缓解示意')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ZNE_error_mitigation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n有噪声(λ=1): {noisy_energies[0]:.6f} Ha (误差 {abs(noisy_energies[0]-E_true)*1000:.2f} mHa)")
print(f"ZNE线性外推: {E_zne_linear:.6f} Ha (误差 {abs(E_zne_linear-E_true)*1000:.2f} mHa)")
print(f"ZNE二次外推: {E_zne_quad:.6f} Ha (误差 {abs(E_zne_quad-E_true)*1000:.2f} mHa)")

---
## 6. 激发态计算：qEOM-VQE

化学研究通常不只关心基态——激发态（光谱、反应路径、OLED发光）同样重要！

**量子方程运动法（qEOM）**：
$$E_k = E_0 + \omega_k, \quad \text{其中} \omega_k \text{来自求解广义本征值问题} [M]\mathbf{x} = \omega [Q]\mathbf{x}$$

矩阵元素通过在VQE基态上进行量子测量获得。

**其他激发态方法**：
- **折叠倒谱法（Folded Inverse）**：通过修改哈密顿量把激发态变成"新基态"
- **多参考VQE**：针对近简并态（多个低能态）
- **SSVQE（子空间VQE）**：同时优化多个激发态

---
## 7. 本节总结：核心算法对比

```
当前（NISQ时代，2024-2027）：
  VQE → ADAPT-VQE → SQD/QSCI → (Krylov-VQE)
  ↑ 浅电路    ↑ 自适应   ↑ 采样    ↑ 可证明收敛

未来（容错QC，2030+）：
  QPE（相位估计）→ 精确能量
  Quantum Simulation（Hamiltonian模拟）
```

**你需要重点掌握的**（按优先级）：
1. VQE（理论基础，当前主流）
2. UCCSD Ansatz（化学精度关键）  
3. SQD/QSCI（目前最接近实用，详见Phase 3）
4. QPE（未来方向，先理解原理）

**下一步**：`02_Qiskit_Nature_H2_LiH.ipynb` — 用Qiskit Nature自动化以上所有步骤！